# 0AD-Bench OSS Model Comparison

This notebook compares recent compact open-weight instruction models on the 15-probe symbolic model-compatibility suite. The probes cover economy, building and technology, scouting, combat, and adaptive decisions.

The run checks model loading, structured observation use, RTS decision selection, strict macro-action JSON, latency, and token use. These are development probes. They are separate from the official engine-backed 50-task benchmark and full-game results.

## Runtime

Select an L4 GPU in Colab. Models are loaded and released one at a time. Each selected checkpoint fits in 24 GB VRAM using its published BF16 or FP8 weights.

In [ ]:
# @title Install the benchmark and model runtime
import os, subprocess, sys
REPO = '/content/markov-chainsaw'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', '0ad-bench-v0.1', 'https://github.com/ritwikraha/markov-chainsaw.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=5.0', 'accelerate>=1.2', 'sentencepiece', 'wandb'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}/0ad-bench'], check=True)

In [ ]:
# @title Read credentials without displaying them
HF_TOKEN = os.environ.get('HF_TOKEN')
WANDB_API_KEY = os.environ.get('WANDB_API_KEY')
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_WRITE_ACCESS')
    WANDB_API_KEY = WANDB_API_KEY or userdata.get('WANDB_KEY')
except Exception:
    pass
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face access:', 'available' if HF_TOKEN else 'public models only')
print('Weights & Biases logging:', 'available' if WANDB_API_KEY else 'disabled')

## Model matrix

| Family | Checkpoint | Published size or storage | L4 plan |
|---|---|---:|---|
| Gemma 4 | `google/gemma-4-E2B-it` | 10.3 GB repository | BF16 |
| Qwen3.5 | `Qwen/Qwen3.5-4B` | 4B parameters | BF16 |
| Ministral 3 | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.4B language model plus 0.4B vision encoder | BF16 |
| Phi-4 | `microsoft/Phi-4-mini-instruct` | 3.8B parameters, 7.69 GB weights | BF16 |

Official model cards: [Gemma 4 E2B Instruct](https://huggingface.co/google/gemma-4-E2B-it), [Qwen3.5 4B](https://huggingface.co/Qwen/Qwen3.5-4B), [Ministral 3 3B Instruct](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16), and [Phi-4 Mini Instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct).

In [ ]:
# @title Confirm accelerator
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running this notebook.'
gpu = torch.cuda.get_device_properties(0)
print({'gpu': gpu.name, 'vram_gb': round(gpu.total_memory / 2**30, 1), 'torch': torch.__version__})

In [ ]:
# @title Run identical deterministic probes
import gc, json, pathlib, traceback
from zero_ad_bench import BenchmarkRunner
from zero_ad_bench.agents import TransformersJSONAgent
from zero_ad_bench.probes import ProbeEnvironment, default_probe_cases, summarize_probe_results

MODEL_IDS = [
    'google/gemma-4-E2B-it',
    'Qwen/Qwen3.5-4B',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16',
    'microsoft/Phi-4-mini-instruct',
]
OUTPUT = pathlib.Path('/content/oss_model_comparison')
OUTPUT.mkdir(parents=True, exist_ok=True)
all_rows, model_results, failures = [], [], []

for model_id in MODEL_IDS:
    print(f'\nLoading {model_id}')
    safe_name = model_id.replace('/', '--')
    try:
        agent = TransformersJSONAgent(model_id, token=HF_TOKEN, max_new_tokens=96)
        runner = BenchmarkRunner(OUTPUT / 'trajectories' / safe_name)
        summaries = []
        for case in default_probe_cases():
            episode_id = f'{case.id}--{safe_name}'
            summary = runner.run(case.task(), agent, ProbeEnvironment(case), episode_id=episode_id)
            summaries.append(summary)
            all_rows.append(summary)
            print(case.id, 'correct' if summary['success'] else 'incorrect', summary['final_observation'].get('events'))
        result = summarize_probe_results(summaries)
        result['model'] = model_id
        result['status'] = 'complete'
        model_results.append(result)
        print(json.dumps(result, indent=2))
    except Exception as exc:
        failure = {'model': model_id, 'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc)[:500]}
        failures.append(failure)
        model_results.append(failure)
        print(json.dumps(failure, indent=2))
        traceback.print_exc(limit=2)
    finally:
        if 'agent' in locals():
            agent.close()
            del agent
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# @title Save results and print the comparison table
import csv, datetime, shutil
completed = sorted((r for r in model_results if r.get('status') == 'complete'), key=lambda r: (-r['accuracy'], r['mean_latency_ms']))
fieldnames = ['rank', 'model', 'correct', 'probe_count', 'accuracy', 'legal_action_rate', 'mean_latency_ms', 'total_tokens', 'status']
with (OUTPUT / 'leaderboard.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for rank, row in enumerate(completed, 1):
        writer.writerow({key: row.get(key, rank if key == 'rank' else '') for key in fieldnames})
with (OUTPUT / 'results.json').open('w') as handle:
    json.dump({'created_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'hardware': gpu.name, 'models': model_results, 'episodes': all_rows, 'failures': failures}, handle, indent=2)
lines = [
    '# OSS model comparison results', '',
    'Scope: 15 deterministic symbolic model-compatibility probes. These are separate from engine-backed 0 A.D. scores.', '',
    '| Rank | Model | Correct | Accuracy | Legal actions | Mean latency | Tokens |',
    '|---:|---|---:|---:|---:|---:|---:|',
]
for rank, row in enumerate(completed, 1):
    lines.append(f"| {rank} | `{row['model']}` | {row['correct']}/{row['probe_count']} | {row['accuracy']:.1%} | {row['legal_action_rate']:.1%} | {row['mean_latency_ms']:.0f} ms | {row['total_tokens']} |")
if failures:
    lines += ['', '## Load or runtime failures', ''] + [f"- `{r['model']}`: {r['error_type']}: {r['error']}" for r in failures]
(OUTPUT / 'RESULTS.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
archive = shutil.make_archive('/content/oss_model_comparison', 'zip', OUTPUT)
print('Artifact archive:', archive)

In [ ]:
# @title Optional Weights & Biases summary logging
if WANDB_API_KEY and completed:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    run = wandb.init(project='0ad-bench', name='oss-model-comparison-v0.1', config={'models': MODEL_IDS, 'probe_count': 15, 'hardware': gpu.name})
    table = wandb.Table(columns=['model', 'accuracy', 'legal_action_rate', 'mean_latency_ms', 'total_tokens'])
    for row in completed:
        table.add_data(row['model'], row['accuracy'], row['legal_action_rate'], row['mean_latency_ms'], row['total_tokens'])
    run.log({'comparison': table})
    run.finish()
    print('Weights & Biases run logged.')
else:
    print('Weights & Biases logging skipped.')